# PBL_02 不良箇所自動検出 良否判定モデル構築用サンプルコード(池田バージョン)
```
DXQuest_PBL02
│  PBL02_sample_code_1.1.ipynb
│  参考) データ説明.txt
│  train_master.tsv
│  sample_submit.tsv
│
└─train
│   └─regular
│   │      regular_000.jpeg~regular_099
│   │  
│   └─potato
│   │      potato_000.jpeg~potato_102.jpeg
│   │  
│   └─horn
│   │      horn_000.jpeg~horn_056.jpeg
│   │  
│   └─bridge
│          bridge_000.jpeg~bridge_029.jpeg
│  
└─test
│    000.jpeg ~ 212.jpeg
│
└─weights

```



# 実施すること
- 目的
horn, potato, bridge, regularの4分類を行う
- 手段
VGG16を用いて画像分類する
- 実施すること（以下の番号）
1. VGGモデルのダウンロード
1. 目的に合った出力層への変更（〇〇分類から4分類に変更）
1. VGG16のパラメータの固定
追加した最後の3層だけを学習させる。VGG16のほとんどのパラメータは固定する
1. VGG16に入力可能なサイズに画像を小さくする設定
1. 画像の学習は小分けにしたないとできないので（メモリーが足りない）、少しずつ画像を取り出す関数を設定
1. 学習
1. 学習過程の確認
1. testデータの推論
1. 様々な判断指標で結果を判断





# 1. Googleドライブと接続
学習データなどを読み込めるようにするため、Googleドライブとサンプルコードを接続します。

In [ ]:
# Google Driveに接続できるdriveというモジュールをgoogle.colabというライブラリからインポートする。
from google.colab import drive
drive.mount('/content/drive')

# 必要なライブラリのインストール

In [ ]:
!pip install keras==2.15.0
!pip install tensorflow==2.15.0

# google driveとの連結

In [ ]:
# 今回使うライブラリをインポートする
import os #ファイルの操作ができるライブラリ
import cv2 #画像処理をするライブラリ
import numpy as np #大量の数字の計算が高速にできるライブラリをnpとしてインポートする
import pandas as pd #表形式のデータを扱うためのライブラリをpdとしてインポートする
import matplotlib.pyplot as plt #グラフを描画するためのライブラリをpltとしてインポートする
from PIL import Image #画像処理のためのライブラリPILから画像の読み込みや保存をするImageモジュールをインポートする
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
) #評価指標を計算するためのライブラリsklearn.metricsから、使用する評価指標の計算モジュールをインポートする

# 機械学習のためのツールをインポートする
import tensorflow as tf # TensorFlowはAIや機械学習のモデルを作るためのライブラリ
from tensorflow import keras # KerasはTensorFlowの中で、簡単にモデルを作るためのモジュール
from keras import optimizers # モデルの学習を助ける最適化アルゴリズム（効率的に学習を進めるやり方）を提供するモジュールをインポートする
from keras.preprocessing.image import ImageDataGenerator # 画像データの前処理や水増し（データオーギュメンテーション）を行うモジュールをインポートする
from keras.models import Sequential, Model # ニューラルネットワークモデルを作るための基本的なツールをインポートする
from keras.layers import Conv2D, MaxPooling2D, AveragePooling2D, Input # 画像データの学習に欠かせない処理（畳み込み層やプーリング層）を作るモジュールをインポートする
from keras.layers import Activation, Dropout, Flatten, Dense # 画像データの学習に欠かせない処理（活性化関数やドロップアウト層、全結合層など）を追加するためのモジュールをインポートする
from keras import backend as K # Kerasが裏で使っている仕組みにアクセスし、高度な設定やカスタマイズを行うためのモジュールをインポートする

# from tensorflow.keras.utils import np_utils # クラスの名前を数字のリストに変えるためのモジュールをインポートする
from keras.applications.vgg16 import VGG16 # 事前に学習されたVGG16モデルをインポートし、それを使って学習を進める（転移学習）
tf.random.set_seed(1) # 実験の結果が毎回同じになるようにする（乱数シードを固定する）
plt.style.use('ggplot') # グラフのスタイルを「ggplot」に設定して、視覚的に見やすくする

# 学習条件の設定

In [ ]:
# 画面サイズの設定
IMG_WIDTH, IMG_HEIGHT = 224, 224
TARGET_SIZE = (IMG_WIDTH, IMG_HEIGHT)

# 分類数（potato,horn,bridge, regularの四つ）
NB_CLASSES = 4

# すべてのデータを学習したとき、1エポック。30エポックでは30回学習する。
# 少ないと、未学習、多すぎると過学習（trainデータだけに有効なモデルになること）になる
EPOCHS = 30

# 一度にすべての画像を学習させることができない（メモリーが膨大に必要）
# そこで小分けにして学習させる。その小分けにした時の画像の枚数
# 基本的にはGPUメモリーに乗るなるべく大きなバッチ数を選択する　⇒　汎化性能がでやすい
BATCH_SIZE = 5

In [ ]:
# データの形を判定する（色のデータが最初か最後か判定している）
if K.image_data_format() == 'channels_first':
    input_shape = (3, IMG_WIDTH, IMG_HEIGHT)
else:
    input_shape = (IMG_WIDTH, IMG_HEIGHT, 3)

In [ ]:
# 学習データ保存場所のパスの設定
train_data_dir = '/content/drive/MyDrive/DXQuest_PBL02/train'

# 検証用データ保存場所のパスの設定
validation_data_dir = '/content/drive/MyDrive/DXQuest_PBL02/train'

# テストデータ保存場所のパスの設定
test_data_dir = '/content/drive/MyDrive/DXQuest_PBL02/test'

# # モデルの保存場所・・・A
# weight_dir = '/content/drive/MyDrive/DXQuest_PBL02/weights'

# # weightファイルの名前・・・B
# save_weights_path = os.path.join(weight_dir, 'weights.weights.h5') # 'weights.h5'のファイル名は変更可

# 上のコメントアウト（A,B）は以下でOKです。（わかりにくいじゃん）
save_weights_path =  '/content/drive/MyDrive/DXQuest_PBL02/weights/weights.weights.h5'

# 学習済みモデルのダウンロードと改造
ここでは「imagenet」を用いて、IMG_WIDTH(224), IMG_HEIGHT(224)を設定する

In [ ]:
# VGG16という画像分類アルゴリズムの基本モデルの初期設定をする
base_model = VGG16(
    # 学習済みの重みを利用する。'imagenet'は、画像データベースImageNetで事前に学習された重みを使う設定。
    weights='imagenet',

    # モデルの最上位の全結合層（分類を行う部分）を除外する設定。この後、自分で分類層を追加してカスタマイズするために使う。
    include_top=False,

    # モデルの入力サイズを指定する。ここでは、224x224ピクセルのRGB画像を入力として受け取るように設定している。
    input_tensor=Input(shape=(IMG_WIDTH, IMG_HEIGHT, 3))
)

In [ ]:
# ネットワーク構造の確認
base_model.summary()

# ダウンロードしたVGG16の出力層だけを取り出して、4分類するためにカスタマイズする
もともとは〇〇種類の分類のモデル構造になっている

In [ ]:
# 出力層だけを取り出す
top_model = base_model.output

# 出力は〇〇次元なので、1次元に変換する
top_model = Flatten(name='flatten')(top_model)

# 必要な情報を抽出するための層を追加する（ニューロン数512、活性化関数のタイプがReLU
top_model = Dense(512, activation='relu')(top_model)

# 過学習防止のために50%のニューロンをランダムに無効にする。
top_model = Dropout(0.5)(top_model)

# 最後の出力層を作る。4分類するので、NB_CLASS（中身は4）を入れ
# softmaxは4出力を確立に変換する関数（４っつ足し合わせたら1になる）
top_model = Dense(NB_CLASSES, activation='softmax')(top_model)

# 上でカスタマイズした出力層をもとのモデルに合体する

In [ ]:
model = Model(
    # モデルの入力部分を設定する。VGG16の入力部分をそのまま使う。
    inputs=base_model.input,
    # モデルの出力部分を設定する。先ほどカスタマイズした出力層をここで使う。
    outputs=top_model
)


# 追加した出力層のみを学習させたいので、もともとあるパラメータを固定する

In [ ]:

# ベースモデル（VGG16）の各層を繰り返し処理する。
for layer in base_model.layers:

    # ベースモデルの層を固定（凍結）して、学習中に重みが更新されないようにする。
    # これにより、事前学習された特徴をそのまま利用し、追加した層のみを学習させることができる。
    layer.trainable = False

# 学習方法の定義

In [ ]:
# VGGモデルの学習方法を定義する
model.compile(
    # 出力と正解の誤差を計算するため損失関数の設定：4分類なので'categorical_crossentropy'を使う
    loss='categorical_crossentropy',

    # パラメータの更新方法を指定（ここではRMSpropという方法を選択し、学習率（learning_rate）が0.0001
    optimizer=keras.optimizers.RMSprop(learning_rate=1e-4),

    # モデルの学習の指標の設定。ここでは'accuracy'（正解率）を指定
    metrics=['accuracy'],
)

In [ ]:
# ネットワーク構造の確認
model.summary()

# 画像サイズの縮小
VGG16に入力可能なサイズを224,224に設定したが、元画像は4032×3024で入力できない
そこで画像のサイズ変換を行う

In [ ]:
# VGGに入力するための画像サイズに圧縮
train_datagen = ImageDataGenerator(rescale=1.0/255) # 前処理を（）内に追加可能
valid_datagen = ImageDataGenerator(rescale=1.0/255) # 前処理を（）内に追加可能

# 学習、検証用データを効率的に取り出す関数を作る
最初に記載した通り、全データを一度に入れることは不可能（GPUのメモリーが足りない）
そこで、batch単位でデータを取り出し、出力、誤差計算、パラメータ更新を行う
batch単位で取り出すのに便利な関数を設定する。書き方はtensorflow,kerasで決まっているのでま、お作法と思ってください。

In [ ]:
#学習・検証データの読み込み
# 保存先、画像サイズ、バッチサイズをそれぞれ設定し、学習が偏らないようにシャッフルさせる
train_generator = train_datagen.flow_from_directory(
    train_data_dir,
    target_size=TARGET_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
)

validation_generator = valid_datagen.flow_from_directory(
    validation_data_dir,
    target_size=TARGET_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
)

In [ ]:
# 確認：学習・検証データとして上記で設定した画像の枚数を取得してみる
nb_train_samples = train_generator.samples
nb_validation_samples = validation_generator.samples

# モデルの学習

In [ ]:
model.fit(
    # 学習データの設定。
    train_generator,

    # 全データを学習するのに、何バッチ必要かを設定
    steps_per_epoch=int(nb_train_samples/BATCH_SIZE),

    # 全データを何度学習させるかを指定（エポック数）
    epochs=EPOCHS,

    # 学習過程のモデルの性能を確認するため検証データを設定
    validation_data=validation_generator,

    # 検証データをすべて検証するのに、何バッチ必要かを設定
    validation_steps=int(nb_validation_samples/BATCH_SIZE)
)

# 学習済みのモデルを保存

In [ ]:
model.save_weights(save_weights_path)

# 学習過程の表示
Training loss（訓練損失）：学習用データで学習しているときに出力と正解データの誤差  
Validation loss（検証損失）：エポックごとに検証用データ（学習に使っていないデータです）で出力させた時の正解データとの誤差  
  
モデルが「どれだけうまく学習できているか」を評価するのに有効  
  
Training lossだけが下がって、Validation lossと乖離するときは過学習（学習データだけにフィッティングし過ぎ）  
Training loss、Validation lossともに下がっている途中は未学習  
Training loss、Validation lossともに乖離せずに下がりきっている状態がベスト  

In [ ]:
# loss curve（モデルの正確性を表す損失関数の学習過程における変化）の表示
plt.figure(figsize=[10,8]) #グラフの大きさを設定
plt.plot(model.history.history['loss'], 'r') #学習データの損失関数を赤色（'r'）で表示する
plt.plot(model.history.history['val_loss'], 'b') #検証データの損失関数を青色（'b'）で表示する
plt.legend(['Training loss', 'Validation Loss']) #グラフに反例を追加する
plt.xlabel('Epochs', fontsize=16) #横軸のラベルを設定する
plt.ylabel('Loss', fontsize=16) #縦軸のラベルを設定する
plt.title('Loss Curves', fontsize=16) #グラフのタイトルを設定する

#グラフを表示する
plt.show()

⭐️初学者向け次のコード解説
```
このコードは、モデルの「正確性（Accuracy）」が時間の経過とともにどう変化していったかを示すグラフを描くためのものです。

・赤い線（学習データの正確性）：モデルが学習する過程で、正解率がどう上がっていったかを示しています。
・青い線（検証データの正確性）：学習後、別のデータを使ってモデルがどれだけ正しく予測できたかを示しています。

このグラフは、モデルが学習を進めるにつれてどれだけ正確に問題を解けるようになったか、またその正確さが新しいデータに対しても
どのように変わっているかを確認するために使います。

＜理想のカーブ＞
・両方の線が徐々に増加：エポックが進むごとに、学習と検証の両方で正確性が上がるのが理想です。
・検証データの正確性が安定：ある時点で検証データの正確性（青い線）が横ばいになっても、それは自然なことです。
モデルがそのデータセットに対して最大の能力を発揮し、これ以上改善しない場合もあります。
```

In [ ]:
# accuracy curve（モデルの正確性の指標の学習過程における変化）の表示
plt.figure(figsize=[10,8])
plt.plot(model.history.history['accuracy'], 'r')
plt.plot(model.history.history['val_accuracy'], 'b')
plt.legend(['Training Accuracy', 'Validation Accuracy'])
plt.xlabel('Epochs', fontsize=16)
plt.ylabel('Accuracy', fontsize=16)
plt.title('Accuracy Curves', fontsize=16)

plt.show()

# 学習したデータを使ってtestデータを分類、判定するための準備

In [ ]:
# weightファイルの読み込み
print('load model...')
model.load_weights(save_weights_path)

## 良否判定を行うための関数

In [ ]:

def get_predict(model,
                train_data_dir: str,
                test_data_dir: str):
    """この関数は、テスト用の画像データを使ってモデルの予測を行い、
    その結果をリストに保存する。

    Args:
        model (object): 訓練済みのモデル。
        train_data_dir (str): 学習用画像が保存されているフォルダのパス。
        test_data_dir (str): テスト用画像が保存されているフォルダのパス。

    Returns:
        filenames (list): 予測した画像のファイル名のリスト。
        true_classes (list): 予測した画像の本当のクラス（カテゴリー）のリスト。
        pred_classes (list): モデルが予測したクラス（カテゴリー）のリスト。
    """

    # テスト用画像を読み込み、モデルに入力するための準備をする
    # 画像の色の値を0から1の範囲に変換して扱いやすくする
    data_datagen = ImageDataGenerator(rescale=1/255.)

    # テスト用の画像を一枚ずつ読み込んでモデルに渡す準備をする
    test_generator = data_datagen.flow_from_directory(
        test_data_dir,
        target_size=TARGET_SIZE,  # 画像の大きさを指定されたサイズに変更する
        class_mode=None,  # ラベル（正解）は指定しない
        batch_size=1,  # 一度に1枚の画像を処理する
        shuffle=False,  # 画像の順番をシャッフルしない（後でファイル名と予測結果を対応させるため）
    )

    # モデルにテスト用の画像を入力して、どのクラスに分類されるかを予測する
    preds = model.predict_generator(test_generator)

    # 予測結果から、一番高い確率で選ばれたクラスの番号を取り出す
    preds_class_idx = preds.argmax(axis=-1)

    # 予測したクラスの名前を取得するための準備
    # 学習用画像の情報を使って、クラス名（カテゴリー名）を取得する
    train_datagen = ImageDataGenerator(rescale=1./255)

    train_generator = train_datagen.flow_from_directory(
        train_data_dir,
        target_size=TARGET_SIZE,
        batch_size=BATCH_SIZE,
    )

    # モデルが使っている番号とクラス名（カテゴリー名）を対応させる辞書を作る
    idx_to_class = {v: k for k, v in train_generator.class_indices.items()}
    # 予測されたクラスの番号をクラス名に変換する
    pred_classes = np.vectorize(idx_to_class.get)(preds_class_idx)
    # ファイル名と予測されたクラス名をペアにしたリストを作成する
    filenames_to_class = list(zip(test_generator.filenames, pred_classes))

    # 本当のクラス名（カテゴリー名）を取得するための準備
    filenames = []
    true_classes = []

    for item in test_generator.filenames:
        filenames.append(item)
        # ファイル名から本当のクラス名（カテゴリー名）を取り出す
        true_class = item.split('/')[0]
        true_classes.append(true_class)

    return filenames, true_classes, pred_classes

## 「どれだけ正確に答えを予測できたか」を評価するために、F1スコア、精度（Precision）、再現率（Recall）という3つの指標を計算する関数

```
F1スコア：予測の全体的なバランスを示す指標。精度と再現率のバランスが良いほどF1スコアが高くなります。
精度（Precision）：コンピュータが「正しい」と予測した中で、本当に正しかったものの割合。
再現率（Recall）：本当に正しかったものを、コンピュータがどれだけ正しく見つけたか。
```

In [ ]:
# 精度算出関数（F1スコア、精度、再現率を計算する関数）
def get_f1(true_labels_list: list,
           predictions_list: list,
           average_method: str,
          ) -> (float, float, float):
    """この関数は、モデルの予測結果に基づいて、F1スコア、精度、再現率を計算する。

    Args:
        true_labels_list (list): 正解ラベルのリスト。
        predictions_list (list): 予測ラベルのリスト。
        average_method (string): スコアを平均化する方法（マルチクラス分類の場合）。

    Returns:
        f1 (float): F1スコアを返す。
        precision (float): 精度を返す。
        recall (float): 再現率を返す。
    """

    # F1スコアを計算する。F1スコアは精度と再現率のバランスをとる指標。
    f1 = f1_score(
        y_true=true_labels_list,  # 正解ラベルのリスト
        y_pred=predictions_list,  # 予測ラベルのリスト
        average=average_method    # 複数クラスの場合にどう平均化するかを指定
    )

    # 精度（Precision）を計算する。精度は、モデルが予測した「正しい」予測の割合を示す指標。
    precision = precision_score(
        y_true=true_labels_list,  # 正解ラベルのリスト
        y_pred=predictions_list,  # 予測ラベルのリスト
        average=average_method,   # 複数クラスの場合にどう平均化するかを指定
    )

    # 再現率（Recall）を計算する。再現率は、実際に正解であったものを、どれだけ正しく予測できたかを示す指標。
    recall = recall_score(
        y_true=true_labels_list,  # 正解ラベルのリスト
        y_pred=predictions_list,  # 予測ラベルのリスト
        average=average_method,   # 複数クラスの場合にどう平均化するかを指定
    )

    # 計算したF1スコア、精度、再現率を少数第2位まで丸める
    f1 = round(f1, 2)
    precision = round(precision, 2)
    recall = round(recall, 2)

    # F1スコア、精度、再現率を返す
    return f1, precision, recall

## 上で設定した二つの関数を用いて、実行
なんで学習データも推論してるんだろ？？？

In [ ]:
# 良否判定実行（学習データに対してモデルの予測を行う）
train_filenames, train_true_classes, train_pred_classes = get_predict(
    model=model,  # 使用するモデルを指定
    train_data_dir=train_data_dir,  # 学習データが保存されているフォルダのパスを指定
    test_data_dir=train_data_dir,  # 学習データをテストデータとして使用し、モデルの予測を実行
)

# 良否判定実行（検証データに対してモデルの予測を行う）
valid_filenames, valid_true_classes, valid_pred_classes = get_predict(
    model=model,  # 使用するモデルを指定
    train_data_dir=train_data_dir,  # 学習データが保存されているフォルダのパスを指定
    test_data_dir=validation_data_dir,  # 検証データをテストデータとして使用し、モデルの予測を実行
)

# 精度評価
【大事】学習・検証データに対する精度をF1-score、Precision、Recallの説明

⭐️初学者向け解説（F1スコア / 精度（precision） / 再現率（recall））
```
・F1スコア
F1スコアは、精度と再現率のバランスを取った指標です。どちらか一方が高くても、もう一方が低いとモデルの性能が十分ではないので、
両方のバランスを考慮して評価するのがF1スコアです。値が1に近いほど良いモデルを意味します。

・精度（precision）
精度は、モデルが「正しい」と予測したものの中で、本当に正しかったものの割合です。
たとえば、100回予測して50回「正しい」と予測したけど、そのうち40回が本当に正しかった場合、精度は40/50 = 80%です。

・再現率（recall）
再現率は、実際に正しいものの中で、モデルが「正しい」と予測できた割合です。
たとえば、実際には50回正しい答えがあったけど、モデルが40回しか正しく予測できなかった場合、再現率は40/50 = 80%です。
```

In [ ]:
# 学習データに対する精度の算出（F1スコア、精度、再現率を計算）
train_f1, train_prec, train_recall = get_f1(
    true_labels_list=train_true_classes,  # 学習データの正解クラスのリストを指定
    predictions_list=train_pred_classes,  # 学習データに対するモデルの予測結果のリストを指定
    average_method='weighted',  # 各クラスの重要度に応じて平均を取る方法を指定（クラスの不均衡がある場合に有効）
)

# 検証データに対する精度の算出（F1スコア、精度、再現率を計算）
valid_f1, valid_prec, valid_recall = get_f1(
    true_labels_list=valid_true_classes,  # 検証データの正解クラスのリストを指定
    predictions_list=valid_pred_classes,  # 検証データに対するモデルの予測結果のリストを指定
    average_method='weighted',  # 各クラスの重要度に応じて平均を取る方法を指定（クラスの不均衡がある場合に有効）
)

## 精度の表示


In [ ]:
# 精度表示
print('{:15}{:<15.2f}{:<15.2f}'.format('F1-score:', train_f1, valid_f1))
print('{:15}{:<15.2f}{:<15.2f}'.format('Precision:', train_prec, valid_prec))
print('{:15}{:<15.2f}{:<15.2f}'.format('Recall:', train_recall, valid_recall))

# 提出ファイルの作成
テストデータに対して良否判定を行い、その結果を提出フォーマットであるtsv形式で出力を行います。

In [ ]:
# Kerasの画像前処理ユーティリティをインポートする
# これにより、画像の読み込みやサイズ変更、データ拡張（オーギュメンテーション）などを簡単に行うことができる。
from keras.preprocessing import image

# ファイル名やパスのパターンマッチングを行うための標準ライブラリをインポートする
# 例えば、特定のフォルダ内のすべての画像ファイル（.jpgや.png）を取得したいときに便利。
import glob

In [ ]:
# 分類とラベル（どの番号がどのクラスか）の対応を確認する
label_map = (train_generator.class_indices)
print(label_map)

In [ ]:
# テストデータに対して1つずつ予測し、ファイル名と判定結果をリストに保存する
file_list = []  # 予測したファイル名を保存するリスト
pred_list = []  # 予測結果を保存するリスト

# テストデータのディレクトリ内のすべてのファイルに対して処理を行う
for file in glob.glob(test_data_dir + '/*'):
    image_data = file  # 現在処理しているファイルのパスを取得
    filename = file.split('/')[-1]  # ファイル名を取得（パスからファイル名だけを取り出す）

    # 画像を指定されたデータ形式に変換する
    img = image.load_img(image_data, target_size=(IMG_WIDTH, IMG_HEIGHT))
    x = image.img_to_array(img)
    x = np.expand_dims(x, axis=0)
    x = x / 255

    # モデルを使って予測を行い、結果を取得する
    pred = model.predict(x)[0]

    # 予測結果の中で最も高い確率を持つクラス（ラベル）を選ぶ
    judge = np.argmax(pred)

    # 予測結果を良品（'0'）か不良（'1'）に変換する
    # *bridge, horn, potatoを不良（'1'）に、regularを良品（'0'）に変換する
    # この条件分岐は、前に確認した「分類とラベルの対応確認」セルの結果を参考にして調整する
    if judge == 0:
        judge = 1  # クラス0（例えばbridge）は不良品と判定
    elif judge == 1:
        judge = 1  # クラス1（例えばhorn）も不良品と判定
    elif judge == 2:
        judge = 1  # クラス2（例えばpotato）も不良品と判定
    else:
        judge = 0  # それ以外のクラス（例えばregular）は良品と判定

    # 判定結果とファイル名をそれぞれリストに追加する
    pred_list.append(judge)
    file_list.append(filename)

---
#####分類とラベルの対応を確認することで、モデルがどのクラスに対してどのラベルを使っているかを理解できました。<br>そして、テストデータの各画像に対して予測を行うことで、その画像が「良品」か「不良品」かを判断し、結果をリストに保存できました。
---

In [ ]:
#判別結果をDataFrameに変換し、tsvファイルに出力
df = pd.DataFrame([file_list, pred_list]).T
df.to_csv('/content/drive/MyDrive/DXQuest_PBL02/IDXXXXXX_PBL02_verXX.tsv',
         index=False,
         header=False,
         sep='\t')

---
#####予測結果が格納されたCSVファイルを「IDXXXXXX_PBL02_verXX.tsv」という名前で保存することができました。<br>ファイルを識別するためにファイル名を変更しておきましょう。<br>(例：ID123456_PBL02_ver1.tsv)<br>まだこれまでに一度も提出していない方は、こちらのファイルを一回目の投稿としてAIモデル精度評価サイトに提出しましょう。<br>
---

###AIモデル精度評価サイト　https://pbl-eval.life-is-tech.com/